
## Gold Layer:  Dimensional Modelling

**Objective:** Transform validated Silver datasets into business-ready dimensional and fact models for air-quality analysis, sensor reliability, and downstream dbt marts.

## Architecture

OpenAQ API → Raw JSON → Bronze Delta → Silver Delta → Gold Delta → dbt Marts → Power BI

**Grain design:**
- Sensor-level daily fact: one sensor × pollutant × monitoring-location local date.
- Location-level daily metric: one location × pollutant × local date, derived using an explicit sensor aggregation policy.

**Engineering requirements:**
- Preserve source lineage and sensor identity.
- Validate parameter units and observation intervals.
- Distinguish observation validity from daily coverage.
- Preserve incomplete sensor-days for quality analysis.
- Prevent duplicate observations and accidental double-counting.
- Keep reporting thresholds configurable.

**Workflow:** Inspect Silver contracts → design dimensions and facts → implement notebook transformations → validate → persist Delta → refactor for deployment.

**Packaging:** The full Gold write path — Silver reads, input schema verification, Type 1 dimensions, `dim_date`, `dim_location` SCD Type 2, `fact_measurements`, and every validation/assert — now lives in `spark/transformations/gold.py` (`build_gold`). This notebook is a thin runner; one-off exploration and investigation cells were removed.


## 1. Setup

Add the bundle-deployed `spark/` package to the path, import the packaged Gold builder, and pin the catalog.

In [ ]:
import sys

PACKAGE_ROOT = "/Workspace/Users/dammifabz@gmail.com/.bundle/openaq-databricks/dev/files"  # bundle-deployed spark/ package; adjust for exploration
if PACKAGE_ROOT not in sys.path:
    sys.path.append(PACKAGE_ROOT)

from spark.transformations.gold import build_gold

CATALOG = "workspace"


## 2. Build and validate the Gold layer

`build_gold` reads the persisted Silver tables, builds and merges all dimensions (Type 1, `dim_date`, `dim_location` SCD Type 2), builds and merges `fact_measurements`, runs every validation/assert from the original notebook, and returns the persisted row count per written table.

In [ ]:
counts = build_gold(spark, CATALOG)

print("Persisted Gold row counts:")
for table_name, row_count in counts.items():
    print(f"{table_name}: {row_count:,} rows")